# PAG-Vul cross-benchmark generalization
Five existing source checkpoints per method are evaluated on each complete unseen target benchmark.

In [ ]:
!pip install -q torch-geometric pennylane


In [ ]:
import shutil
import subprocess
import torch
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/generalization_results')
ROOTS = list(INPUT.iterdir())
if not ROOTS:
    import kagglehub
    ROOTS = [Path(kagglehub.dataset_download(handle)) for handle in (
        'khangtrn2/benchmarkpython-pagvul-binary-assets',
        'khangtrn2/vudenc-pagvul-binary-assets',
        'khangtrn2/realvuln-human-pagvul-binary-assets',
        'khangtrn2/pagvul-cross-benchmark-generalization-assets',
    )]

def one(name):
    matches = [path for root in ROOTS for path in root.rglob(name)]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {name}, found {matches}')
    return matches[0]

if not torch.cuda.is_available():
    raise RuntimeError('A compatible Kaggle GPU is required')
torch.zeros(1, device='cuda').add_(1).item()
evaluator = one('evaluate_cross_dataset.py')
runs_root = evaluator.parent / 'runs'
if not runs_root.is_dir():
    archive = evaluator.parent / 'runs.zip'
    if not archive.is_file():
        raise RuntimeError(f'Missing checkpoint tree and archive beside {evaluator}')
    unpacked = Path('/kaggle/working/generalization_checkpoints')
    shutil.unpack_archive(archive, unpacked)
    runs_root = unpacked / 'runs' if (unpacked / 'runs').is_dir() else unpacked
graphs = {
    'benchmarkpython': one('benchmarkpython_binary_graphs.pt'),
    'vudenc': one('vudenc_binary_graphs.pt'),
    'realvuln_human': one('realvuln_human_binary_graphs.pt'),
}
command = ['python', str(evaluator)]
for dataset, path in graphs.items():
    command += ['--graph', f'{dataset}={path}']
command += ['--runs-root', str(runs_root), '--output', str(WORK), '--device', 'cuda', '--graph-batch-size', '64']
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
print((WORK / 'summary.md').read_text())
